# Test Lagged Features

Test whether previous month's delay rate helps predict current month's delay rate.

**Hypothesis:** Delays exhibit temporal autocorrelation - airlines with delays last month likely have delays this month (operational issues persist, seasonal patterns, etc.)

**Features to test:**
- `delay_rate_lag1`: Previous month's delay rate (same airline-route)
- `delay_rate_lag2`: 2 months ago
- `delay_rate_lag3`: 3 months ago
- Rolling averages (3-month, 6-month)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Load and Prepare Data

In [ ]:
# Load data
df = pd.read_csv('../data/processed/ml_training_data_syd_mel.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = pd.to_datetime(df['month']).dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']

print(f"Shape: {df.shape}")
print(f"Unique airline-routes: {df['airline_route'].nunique()}")

In [ ]:
# Sort by airline-route and date for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)
df.head(10)

## 2. Create Lagged Features

In [ ]:
# Create lagged features (3 new columns) within each airline-route group
# Shifted by 1, 2 and 3 rows
for lag in [1, 2, 3]:
    df[f'delay_rate_lag{lag}'] = df.groupby('airline_route')['delay_rate'].shift(lag)

# Also try rolling averages (previous 3 and 6 months)
df['delay_rate_rolling3'] = df.groupby('airline_route')['delay_rate'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
df['delay_rate_rolling6'] = df.groupby('airline_route')['delay_rate'].transform(
    lambda x: x.shift(1).rolling(window=6, min_periods=1).mean()
)

# Print some columns to inspect
print("Lagged features created:")
print(df[['airline_route', 'year_month', 'delay_rate', 'delay_rate_lag1', 
          'delay_rate_lag2', 'delay_rate_rolling3']].head(15))

In [ ]:
# Check missing values (first few months will have NaN for lags)
lag_cols = ['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_lag3', 
            'delay_rate_rolling3', 'delay_rate_rolling6']
print("Missing values in lagged features:")
print(df[lag_cols].isnull().sum())

In [ ]:
# Drop rows with missing lag values
df_clean = df.dropna(subset=['delay_rate_lag1']).copy()
print(f"Rows after dropping NaN: {len(df_clean)} (dropped {len(df) - len(df_clean)})")

In [ ]:
# Visualize autocorrelation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, lag in enumerate([1, 2, 3]):
    col = f'delay_rate_lag{lag}'
    ax = axes[i]
    
    # Drop NaN values for this specific lag
    mask = df_clean[col].notna() & df_clean['delay_rate'].notna()
    x_data = df_clean.loc[mask, col].values
    y_data = df_clean.loc[mask, 'delay_rate'].values
    
    ax.scatter(x_data, y_data, alpha=0.3, s=10)
    
    # Trend line - use only valid data
    if len(x_data) > 2:
        z = np.polyfit(x_data, y_data, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line, p(x_line), 'r-', linewidth=2)
    
    r = np.corrcoef(x_data, y_data)[0, 1]
    ax.set_xlabel(f'Delay Rate (t-{lag})')
    ax.set_ylabel('Delay Rate (t)')
    ax.set_title(f'Lag {lag}: r = {r:.3f}')

plt.suptitle('Autocorrelation of Delay Rate with Lag', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of lagged features with target
print("Correlation with delay_rate:")
print("=" * 40)
for col in lag_cols:
    r = df_clean[col].corr(df_clean['delay_rate'])
    print(f"  {col:<25}: {r:.4f}")

Delay rate with 1 month lag looks promising, with an auto-correlation value of almost 0.7.

In [ ]:
# Prepare features
# Cyclical month encoding
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month_num'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df_clean['airline'], prefix='airline')
df_clean = pd.concat([df_clean, airline_dummies], axis=1)

# Get only the one-hot encoded columns (not airline_route)
airline_cols = [c for c in airline_dummies.columns]
print(f"Airline columns: {airline_cols}")

## 4. Model Comparison

In [ ]:
# Prepare features
# Cyclical month encoding
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month_num'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df_clean['airline'], prefix='airline')
df_clean = pd.concat([df_clean, airline_dummies], axis=1)

# Get only the one-hot encoded columns (not airline_route)
airline_cols = [c for c in airline_dummies.columns]
print(f"Airline columns: {airline_cols}")

In [ ]:
# Define feature sets
baseline_features = airline_cols + ['month_sin', 'month_cos']
lag1_features = baseline_features + ['delay_rate_lag1']
all_lag_features = baseline_features + ['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_lag3',
                                         'delay_rate_rolling3', 'delay_rate_rolling6']

print(f"Baseline features: {len(baseline_features)}")
print(f"With lag1: {len(lag1_features)}")
print(f"All lags: {len(all_lag_features)}")

In [ ]:
# Time-based split
# Excluding 2020-2022 (COVID period)

train_mask = ((df['year'] <= 2017) | (df['year'] == 2024))
val_mask = ((df['year'] == 2018) | (df['year'] == 2023))
test_mask = ((df['year'] == 2019) | (df['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025)     : {test_mask.sum()} samples")
print("")
print("Note: 2020-2022 excluded (COVID period)")

In [ ]:
# Prepare data for each feature set
def prepare_data(features):
    # Remove rows with NaN in selected features
    mask = df_clean[features].notna().all(axis=1)
    df_subset = df_clean[mask]
    
    train = df_subset[(df_subset['year'] <= 2017) | (df_subset['year'] == 2024)]
    val = df_subset[(df_subset['year'] == 2018) | (df_subset['year'] == 2023)]
    test = df_subset[(df_subset['year'] == 2019) | (df_subset['year'] >= 2025)]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[features])
    X_val = scaler.transform(val[features])
    X_test = scaler.transform(test[features])
    
    return (X_train, train['delay_rate'].values, train['is_high_delay'].values,
            X_val, val['delay_rate'].values, val['is_high_delay'].values,
            X_test, test['delay_rate'].values, test['is_high_delay'].values)

In [ ]:
def evaluate_model(features, name):
    """Train and evaluate regression model."""
    X_train, y_train_reg, y_train_clf, X_val, y_val_reg, y_val_clf, X_test, y_test_reg, y_test_clf = prepare_data(features)
    
    # Ridge regression
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train_reg)
    
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    results = {
        'name': name,
        'val_rmse': np.sqrt(mean_squared_error(y_val_reg, y_val_pred)),
        'val_r2': r2_score(y_val_reg, y_val_pred),
        'test_rmse': np.sqrt(mean_squared_error(y_test_reg, y_test_pred)),
        'test_r2': r2_score(y_test_reg, y_test_pred),
    }
    
    # Classification
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    clf.fit(X_train, y_train_clf)
    
    y_test_pred_clf = clf.predict(X_test)
    y_test_proba_clf = clf.predict_proba(X_test)[:, 1]
    
    results['test_f1'] = f1_score(y_test_clf, y_test_pred_clf)
    results['test_auc'] = roc_auc_score(y_test_clf, y_test_proba_clf)
    
    return results

In [ ]:
# Evaluate all feature sets
results = []
results.append(evaluate_model(baseline_features, 'Baseline (airline+month)'))
results.append(evaluate_model(lag1_features, 'Baseline + Lag1'))
results.append(evaluate_model(all_lag_features, 'Baseline + All Lags'))

# Also test lag-only model
results.append(evaluate_model(['delay_rate_lag1'], 'Lag1 Only'))

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Display comparison
print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(f"\n{'Model':<30} {'Val R²':>10} {'Test R²':>10} {'Test F1':>10} {'Test AUC':>10}")
print("-" * 80)

for _, row in results_df.iterrows():
    print(f"{row['name']:<30} {row['val_r2']:>10.4f} {row['test_r2']:>10.4f} {row['test_f1']:>10.4f} {row['test_auc']:>10.4f}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# R² comparison
ax = axes[0]
x = range(len(results_df))
ax.bar(x, results_df['test_r2'], color='steelblue', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(results_df['name'], rotation=30, ha='right')
ax.set_ylabel('Test R²')
ax.set_title('Regression Performance')
ax.axhline(0, color='red', linestyle='--', label='Predict mean baseline')
ax.legend()

# F1 comparison
ax = axes[1]
ax.bar(x, results_df['test_f1'], color='#2ecc71', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(results_df['name'], rotation=30, ha='right')
ax.set_ylabel('Test F1')
ax.set_title('Classification Performance')

plt.tight_layout()
plt.show()

## 5. Summary

In [ ]:
# Calculate improvement
baseline_r2 = results_df[results_df['name'] == 'Baseline (airline+month)']['test_r2'].values[0]
lag1_r2 = results_df[results_df['name'] == 'Baseline + Lag1']['test_r2'].values[0]
all_lag_r2 = results_df[results_df['name'] == 'Baseline + All Lags']['test_r2'].values[0]

baseline_f1 = results_df[results_df['name'] == 'Baseline (airline+month)']['test_f1'].values[0]
lag1_f1 = results_df[results_df['name'] == 'Baseline + Lag1']['test_f1'].values[0]
all_lag_f1 = results_df[results_df['name'] == 'Baseline + All Lags']['test_f1'].values[0]

print("=" * 70)
print("SUMMARY: LAGGED FEATURES IMPACT")
print("=" * 70)

print(f"\nREGRESSION (R²):")
print(f"  Baseline:           {baseline_r2:+.4f}")
print(f"  + Lag1:             {lag1_r2:+.4f} (improvement: {lag1_r2 - baseline_r2:+.4f})")
print(f"  + All Lags:         {all_lag_r2:+.4f} (improvement: {all_lag_r2 - baseline_r2:+.4f})")

print(f"\nCLASSIFICATION (F1):")
print(f"  Baseline:           {baseline_f1:.4f}")
print(f"  + Lag1:             {lag1_f1:.4f} (improvement: {lag1_f1 - baseline_f1:+.4f})")
print(f"  + All Lags:         {all_lag_f1:.4f} (improvement: {all_lag_f1 - baseline_f1:+.4f})")

print("\n" + "=" * 70)
print("VERDICT")
print("=" * 70)

if lag1_r2 > 0.1:
    print("\n✓ LAGGED FEATURES HELP SIGNIFICANTLY")
    print("  Previous month's delay rate is a strong predictor.")
    print("  Consider building a forecasting model with lagged features.")
elif lag1_r2 > baseline_r2 + 0.1:
    print("\n⚡ LAGGED FEATURES PROVIDE MODERATE IMPROVEMENT")
    print("  Some predictive value, but still limited.")
else:
    print("\n⚠️ LAGGED FEATURES DON'T HELP MUCH")
    print("  Delay rates may be too volatile to predict even with history.")

### Observations

**Lagged features help a lot:**
- Delays have temporal persistence (operational issues carry over)
- Next month forecasting is possible(?)
- Consider time-series approaches rather than regression & classification in the future

## 6. Next step

Build a baseline ML model that incorporates the strongest features so far:
- Lagged feature (1 lag)
- Time of the year (month encoded cyclical-ly)
- Maybe sectors flown (more flights crammed -> higher chance of delay)

See [4b_baseline_model_single_route.ipynb](//notebooks/4b_baseline_model_single_route.ipynb)